<a href="https://colab.research.google.com/github/yenlung/Python-Math-AI/blob/main/11%E7%B7%9A%E6%80%A7%E8%BF%B4%E6%AD%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11 線性迴歸 📈
## 機器學習的起點 × 用數據預測未來

終於來到機器學習！

機器學習聽起來很複雜，但其實最核心的思想很簡單：

> **讓電腦從數據中「學習」規律，然後用來預測新的情況。**

而最基礎的機器學習模型，就是你國中就學過的：

# 👉 $y = ax + b$

這就是**線性迴歸**！

---

### 這週你會學到：

- 什麼是「訓練」一個模型
- `scikit-learn` 的三步驟：**打開 → 訓練 → 預測**
- 如何評估模型好不好
- 用真實的加州房價數據做預測
- 多變數線性迴歸

## 🤖 AI 可以怎麼幫你？

你可以問 AI：

- 線性迴歸是什麼？
- `sklearn` 的 `fit`、`predict` 是什麼意思？
- `R²` 分數怎麼解讀？
- 什麼是訓練集和測試集？
- 如果數據不是線性的怎麼辦？

但記得：**先自己想，再問 AI**

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

## 1. 線性迴歸的直覺

想像你在觀察「讀書時間」和「考試分數」的關係。

先來模擬一批帶有雜訊的數據（真實數據就是這樣）：

In [ ]:
np.random.seed(42)
x = np.linspace(0, 5, 100)          # 讀書時間（小時）
y = 1.2*x + 0.3 + 0.3*np.random.randn(100)  # 分數（有雜訊）

plt.scatter(x, y, alpha=0.7)
plt.xlabel('讀書時間（小時）')
plt.ylabel('考試分數')
plt.title('讀書時間 vs 考試分數')
plt.show()

我們想找到一條**最佳擬合直線** $y = ax + b$，讓這條線盡可能「解釋」這些散點。

機器學習的任務是：**從這些數據自動找出最好的 $a$ 和 $b$**，不用我們手動猜！

## 2. scikit-learn 的黃金三步驟

`scikit-learn`（簡稱 `sklearn`）是 Python 最著名的機器學習套件。

不管什麼模型，都是同樣三個步驟：

```
Step 1: model = Model()          ← 打開一個空白學習機
Step 2: model.fit(X, y)          ← 用數據訓練（學習）
Step 3: model.predict(X_new)     ← 預測新數據
```

這個流程在整個機器學習領域都是一樣的！

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# sklearn 要求輸入是 2D 矩陣（每筆數據是一列）
# x 是 (100,) → reshape 成 (100, 1)
x_train = x.reshape(-1, 1)
y_train = y.reshape(-1, 1)

print('x_train.shape:', x_train.shape)  # 100 筆數據，每筆 1 個特徵
print('y_train.shape:', y_train.shape)  # 100 筆標籤

In [ ]:
# Step 1: 打開空白函數學習機
model = LinearRegression()
print('模型建立完成！（還沒學任何東西）')

In [ ]:
# Step 2: 用數據訓練
model.fit(x_train, y_train)
print('訓練完成！')
print(f'學到的斜率 a = {model.coef_[0][0]:.4f}')
print(f'學到的截距 b = {model.intercept_[0]:.4f}')
print(f'（真實值：a=1.2, b=0.3）')

In [ ]:
# Step 3: 預測
print('讀書 1.1 小時的預測分數:')
print(model.predict([[1.1]]))

In [ ]:
# 畫出預測結果
y_pred = model.predict(x_train)

plt.scatter(x_train.ravel(), y_train.ravel(), alpha=0.7, label='實際數據')
plt.plot(x_train.ravel(), y_pred.ravel(), color='red', linewidth=2, label='迴歸線')

plt.xlabel('讀書時間（小時）')
plt.ylabel('考試分數')
plt.title('線性迴歸結果')
plt.legend()
plt.show()

## 3. 真實數據：加州房價預測

剛才用的是「假數據」，現在來挑戰真實世界的問題！

加州房價數據包含 20,000+ 筆房屋資料，特徵包括：
經緯度、屋齡、總房間數、總臥室數、人口、家庭數、收入中位數等。

**任務：預測每個地區的房屋中位價**

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv')

# 把有缺值的數據刪除
df = df.dropna()

print('資料大小:', df.shape)
df.head()

In [ ]:
# 先看看資料的基本統計
df.describe()

In [ ]:
# 房價分佈長什麼樣？
plt.figure(figsize=(8, 4))
plt.hist(df['median_house_value'], bins=50, edgecolor='white')
plt.xlabel('房價中位數（美元）')
plt.ylabel('數量')
plt.title('加州房價分佈')
plt.show()

### 哪個特徵和房價最相關？

在訓練模型前，先用相關係數探索數據！

In [ ]:
# 只看數值欄位的相關係數
numeric_df = df.drop('ocean_proximity', axis=1)
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('特徵相關係數矩陣')
plt.tight_layout()
plt.show()

### 訓練集 vs 測試集

機器學習一個重要原則：

> **不能用來訓練的數據來評估模型！**

就像老師不能用練習題當作考題，要保留「新題目」來測試學生有沒有真的學會。

我們把數據切成：**70% 訓練** + **30% 測試**

In [ ]:
from sklearn.model_selection import train_test_split

# 特徵：從 longitude 到 median_income
X = df.loc[:, 'longitude':'median_income'].values
y = df['median_house_value'].values

print('特徵 X shape:', X.shape)  # 20433 筆，8 個特徵
print('標籤 y shape:', y.shape)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print('訓練集:', x_train.shape)
print('測試集:', x_test.shape)

### 三步驟訓練多變數線性迴歸

In [ ]:
# Step 1
model = LinearRegression()

# Step 2
model.fit(x_train, y_train)

# Step 3
y_pred = model.predict(x_test)

print('預測完成！')
print('前 5 筆預測值:', y_pred[:5].astype(int))
print('前 5 筆真實值:', y_test[:5].astype(int))

## 4. 評估模型好不好

### 方法一：視覺化

預測值 vs 真實值畫在圖上。**完美的模型**會讓所有點落在 $y=x$ 這條紅線上。

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, s=5)
plt.plot([0, 500000], [0, 500000], color='red', linewidth=2, label='完美預測線')
plt.xlabel('真實房價（美元）')
plt.ylabel('預測房價（美元）')
plt.title('線性迴歸預測結果')
plt.legend()
plt.show()

### 方法二：數值指標

| 指標 | 英文 | 說明 | 越好的值 |
|------|------|------|----------|
| 均方誤差 | MSE | 預測誤差的平方平均 | 越小越好 |
| 均方根誤差 | RMSE | MSE 開根號（單位和 y 相同） | 越小越好 |
| 決定係數 | R² | 模型解釋了多少數據變異 | 越接近 1 越好 |

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MSE:  {mse:,.0f}')
print(f'RMSE: {rmse:,.0f} 元（平均誤差約 {rmse/1000:.0f} 千美元）')
print(f'R²:   {r2:.4f}（模型解釋了 {r2:.1%} 的數據變異）')

### 殘差圖：診斷模型的健康狀況

**殘差** = 真實值 − 預測值

好的模型：殘差應該是**隨機分佈**的，不應該有明顯的規律。

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 殘差散佈圖
axes[0].scatter(y_pred, residuals, alpha=0.3, s=5)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('預測值')
axes[0].set_ylabel('殘差')
axes[0].set_title('殘差散佈圖')

# 殘差分佈
axes[1].hist(residuals, bins=50, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_xlabel('殘差')
axes[1].set_ylabel('數量')
axes[1].set_title('殘差分佈')

plt.tight_layout()
plt.show()

print(f'平均殘差: {residuals.mean():.0f}（接近 0 代表模型沒有系統性偏差）')

## 💡 思考與挑戰

### 🤔 為什麼線性迴歸在這個問題上表現不完美？

看看殘差圖，可以發現：
1. 高房價的地方殘差很大（有一個明顯的「天花板」效應）
2. 線性模型可能無法捕捉非線性關係

問 AI：怎麼改進這個模型？（提示：試試 `Ridge`、`RandomForestRegressor`）

In [ ]:
# 挑戰：特徵工程
# 試著加入新特徵，看看能不能提升 R²

df_feat = df.copy()

# 新特徵：每個家庭的平均房間數（可能比總房間數更有意義）
df_feat['rooms_per_household'] = df_feat['total_rooms'] / df_feat['households']
df_feat['bedrooms_per_room'] = df_feat['total_bedrooms'] / df_feat['total_rooms']
df_feat['population_per_household'] = df_feat['population'] / df_feat['households']

# 重新準備特徵
feature_cols = ['longitude', 'latitude', 'housing_median_age',
                'median_income', 'rooms_per_household',
                'bedrooms_per_room', 'population_per_household']

X2 = df_feat[feature_cols].values
y2 = df_feat['median_house_value'].values

x2_train, x2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=42
)

model2 = LinearRegression()
model2.fit(x2_train, y2_train)
y2_pred = model2.predict(x2_test)

r2_new = r2_score(y2_test, y2_pred)
print(f'加入特徵工程後的 R²: {r2_new:.4f}')
print(f'原始 R²: {r2:.4f}')
print(f'改善了: {r2_new - r2:.4f}')

# 🧠 核心觀念回顧

```
機器學習的黃金流程

數據 → 探索 (EDA) → 分割 → 訓練 → 預測 → 評估
                  ↑                          |
                  └────── 特徵工程（改進）────┘
```

| 步驟 | sklearn 用法 |
|------|-------------|
| 分割數據 | `train_test_split(X, y, test_size=0.3)` |
| 建立模型 | `model = LinearRegression()` |
| 訓練 | `model.fit(x_train, y_train)` |
| 預測 | `model.predict(x_test)` |
| 評估 | `r2_score(y_test, y_pred)` |

# 🎯 本週創作任務

找一個**你想預測的真實問題**，用線性迴歸來解決：

- 🏠 **台北租金預測**：坪數、樓層、捷運距離 → 月租金
- 📚 **考試成績預測**：上課出席率、作業完成率 → 期末成績
- 🌡️ **氣溫預測**：日期、地點 → 氣溫
- 💰 **薪資預測**：工作年限、學歷、城市 → 薪資

（可以用 AI 生成模擬數據，或找公開數據集）

回答：

1️⃣ 你選了什麼問題？特徵和標籤是什麼？
2️⃣ 模型的 R² 分數是多少？
3️⃣ 殘差圖看起來如何？模型有沒有改善空間？
4️⃣ AI 幫了你什麼？